# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and analyze the [FAIR² Croissant dataset](https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json) using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL for FAIR² dataset
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# Preview the dataset metadata
metadata = dataset.metadata
print(f"Dataset Title: {metadata.name}")
print(f"Description: {metadata.description}\n")
print(f"FAIR2 identifier: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, columns, and their `@id` values defined in the dataset.

### List Record Sets, Fields, and Columns
We'll print all available record set `@id`s, each field and column `@id`, as described in the Croissant package. This is crucial for referencing components by their stable identifiers.

In [ ]:
# List all record sets defined in the Croissant dataset
print("Record sets in the dataset:")
for record_set in dataset.record_sets:
    print(f"- {record_set['@id']}: {record_set.get('name', '<no name>')}")

# For each record set, list their fields and columns by @id
for record_set in dataset.record_sets:
    print(f"\nRecord Set: {record_set['@id']}")
    print("  Fields:")
    for field in record_set.get('field', []):
        if isinstance(field, dict):
            print(f"    - {field.get('@id', field)}")
        else:
            print(f"    - {field}")
    print("  Columns:")
    for column in record_set.get('column', []):
        if isinstance(column, dict):
            print(f"    - {column.get('@id', column)}")
        else:
            print(f"    - {column}")

## 3. Data Extraction
Load data from a particular record set into a DataFrame for analysis. Always refer to the record set and field/column by their `@id`.

**Below, we'll load records from each record set (replace or add as needed for your exploration).**

In [ ]:
# Collect all record_set @id values
record_set_ids = [r["@id"] for r in dataset.record_sets]
dataframes = {}

# Load DataFrames for each record set
for record_set_id in record_set_ids:
    print(f"Loading records from: {record_set_id}")
    records = list(dataset.records(record_set=record_set_id))
    if records:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"{record_set_id} columns: {df.columns.tolist()}")
        display(df.head())
    else:
        print(f"No records found for {record_set_id}.")

## 4. Exploratory Data Analysis (EDA)
Apply typical processing: filter, normalize, group, and visually inspect numeric fields. All field references use the `@id` string as the column name.

In [ ]:
# Choose which record_set to analyze (usually the main data table)
# Example: pick the first non-empty record_set
if dataframes:
    main_record_set_id = list(dataframes.keys())[0]
    df = dataframes[main_record_set_id]
    print(f"Analyzing record set: {main_record_set_id}")
    
    # Try to find a numeric field for demo (manual or dynamic identify column)
    numeric_candidate = None
    for col in df.columns:
        # Heuristically pick columns with possibly numeric content
        if pd.api.types.is_numeric_dtype(df[col]):
            numeric_candidate = col
            break
    if not numeric_candidate:
        # Try to convert columns to numeric
        for col in df.columns:
            try:
                converted = pd.to_numeric(df[col])
                if converted.notna().sum() > 0:
                    df[col] = converted
                    numeric_candidate = col
                    break
            except Exception:
                continue
    if numeric_candidate:
        print(f"Selected numeric field: {numeric_candidate}")
        threshold = df[numeric_candidate].mean() if df[numeric_candidate].notna().any() else 0
        filtered_df = df[df[numeric_candidate] > threshold]
        print(f"Filtered records with '{numeric_candidate}' > {threshold:.2f}:")
        display(filtered_df.head())

        # Normalize
        filtered_df[f"{numeric_candidate}_normalized"] = (
            filtered_df[numeric_candidate] - filtered_df[numeric_candidate].mean()
        ) / filtered_df[numeric_candidate].std()
        print(f"Normalized '{numeric_candidate}' for filtered records:")
        display(filtered_df[[numeric_candidate, f"{numeric_candidate}_normalized"]].head())

        # Optionally group by another field if one is categorical
        group_field_candidate = None
        for col in df.columns:
            if col != numeric_candidate and df[col].nunique() < 5:
                group_field_candidate = col
                break
        if group_field_candidate:
            print(f"Grouping by: {group_field_candidate}")
            grouped = (
                filtered_df.groupby(group_field_candidate)[numeric_candidate].mean().reset_index()
            )
            display(grouped)
    else:
        print("No numeric field was detected in the main record set.")
else:
    print("No non-empty record sets found to analyze.")

## 5. Visualization
Visualize the distribution of the selected numeric field, if available.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if dataframes and 'numeric_candidate' in locals() and numeric_candidate in df.columns:
    plt.figure(figsize=(7,4))
    sns.histplot(df[numeric_candidate].dropna(), kde=True, bins=30)
    plt.title(f"Distribution of {numeric_candidate} in {main_record_set_id}")
    plt.xlabel(numeric_candidate)
    plt.ylabel("Count")
    plt.show()

## 6. Conclusion
- This notebook demonstrated loading FAIR² Croissant metadata and tabular records using `mlcroissant`.
- All dataset entities (record sets, fields, columns) were referenced by their stable `@id` values.
- Data overview, extraction, filtering, normalization, grouping, and visualization were shown.
- For publication-quality analysis, consult the [dataset documentation](https://sen.science/doi/10.71728/senscience.vq4a-28xa/) for detailed field descriptions and use synthetic thresholds or groupings relevant to your biological questions.